In [0]:
-- One-time fix: set dbt_valid_from to the actual transaction modification date
-- Silver IDs are TXN-format; bronze IDs are integers — normalised via REGEXP_EXTRACT
UPDATE nestle_dev_silver.core.sales_transactions_scd2 AS target
SET dbt_valid_from = (
  SELECT CAST(TO_TIMESTAMP(MAX(b.modified_at)) AS DATE)
  FROM nestle_dev_bronze.sql_server.sales_transactions b
  WHERE TRY_CAST(REGEXP_EXTRACT(b.transaction_id,    '(\\d+)$', 1) AS INT)
      = TRY_CAST(REGEXP_EXTRACT(target.transaction_id,'(\\d+)$', 1) AS INT)
)
WHERE dbt_is_current = TRUE;

-- Verify corrected date distribution
SELECT dbt_valid_from, COUNT(*) as row_count
FROM nestle_dev_silver.core.sales_transactions_scd2
WHERE dbt_is_current = TRUE
GROUP BY dbt_valid_from
ORDER BY dbt_valid_from;

Refresh Fact Table (REPLACE)

In [0]:
-- Refresh Gold Fact Table with Day 1 + Day 2 + Day 3 data
TRUNCATE TABLE nestle_dev_gold.bi_core.f_sales_fact;

INSERT INTO nestle_dev_gold.bi_core.f_sales_fact
SELECT
  scd.transaction_id,
  scd.product_id,
  scd.region,
  scd.channel,
  scd.customer_id,
  scd.quantity,
  scd.unit_price,
  scd.amount,
  scd.dbt_valid_from                    AS transaction_date,
  CAST(scd.dbt_valid_from AS TIMESTAMP) AS created_at
FROM nestle_dev_silver.core.sales_transactions_scd2 scd
WHERE scd.dbt_is_current = TRUE;

SELECT COUNT(*) as fact_rows FROM nestle_dev_gold.bi_core.f_sales_fact;

Refresh MV 1 - Daily Sales

In [0]:
-- Recreate Daily Sales Summary (grouped by date + region only)
CREATE OR REPLACE MATERIALIZED VIEW nestle_dev_gold.bi_core.agg_daily_sales_summary AS
SELECT
  f.transaction_date,
  f.region,
  COUNT(*)          AS transaction_count,
  SUM(f.quantity)   AS total_quantity,
  SUM(f.amount)     AS total_amount,
  AVG(f.amount)     AS avg_transaction_value,
  MIN(f.amount)     AS min_amount,
  MAX(f.amount)     AS max_amount
FROM nestle_dev_gold.bi_core.f_sales_fact f
GROUP BY f.transaction_date, f.region;

SELECT COUNT(*) as daily_agg_rows FROM nestle_dev_gold.bi_core.agg_daily_sales_summary;

Refresh MV 2 - Monthly Revenue

In [0]:
-- Refresh Monthly Revenue by Channel
REFRESH MATERIALIZED VIEW nestle_dev_gold.bi_core.agg_monthly_revenue_by_channel;

SELECT COUNT(*) as monthly_agg_rows FROM nestle_dev_gold.bi_core.agg_monthly_revenue_by_channel;

Refresh MV 3 - Product Performance

In [0]:
-- Refresh Product Performance
REFRESH MATERIALIZED VIEW nestle_dev_gold.bi_core.agg_product_performance;

SELECT COUNT(*) as product_perf_rows FROM nestle_dev_gold.bi_core.agg_product_performance;

Verify All Gold Objects Updated

In [0]:
SELECT 
  (SELECT COUNT(*) FROM nestle_dev_gold.bi_core.f_sales_fact) as fact_table,
  (SELECT COUNT(*) FROM nestle_dev_gold.bi_core.agg_daily_sales_summary) as daily_agg,
  (SELECT COUNT(*) FROM nestle_dev_gold.bi_core.agg_monthly_revenue_by_channel) as monthly_agg,
  (SELECT COUNT(*) FROM nestle_dev_gold.bi_core.agg_product_performance) as product_perf;